# Amazon Review Sentiment Analysis

Educational NLP project comparing Multinomial Naive Bayes and Random Forest using TF-IDF features. Adapted from an existing Colab learning notebook.

Provide your own permitted `Amazon-Product.csv` with `review_body` and binary `sentiment` columns (0 = negative, 1 = positive). Dataset and old outputs are not included. Run the cells in order.

The export fixes train/test vocabulary leakage and duplicate removal. Previous accuracy outputs were cleared; rerun to obtain fresh results. This is a learning experiment, not a production sentiment service.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import nltk
from nltk.tokenize import word_tokenize
from nltk import download
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier


#Load Data

In [ ]:
data = pd.read_csv("Amazon-Product.csv", usecols=["review_body", "sentiment"])
assert data["sentiment"].dropna().isin([0, 1]).all(), "sentiment must contain 0 or 1"


In [ ]:
data[['review_body', 'sentiment']].head()

In [ ]:
data[['review_body', 'sentiment']].tail(10)

In [ ]:
data.shape

In [ ]:
data.describe()

In [ ]:
data.info()

In [ ]:
data.dropna(inplace=True)

In [ ]:
data.drop_duplicates(subset=["review_body"], inplace=True)

In [ ]:
print("Original DataFrame columns:\n", data.columns)


In [ ]:
data.info()

In [ ]:
nltk.download('punkt_tab') # Downloading the missing punkt_tab data
nltk.download('punkt')  # For tokenization using word_tokenize
nltk.download('wordnet')  # For lemmatization
nltk.download('omw-1.4')  # For WordNet lemmatizer data
nltk.download('stopwords')  # For stopwords

In [ ]:
def preprocess_text(text):
    # Convert text to lowercase
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text) ## not word and  string

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Tokenize text
    tokens = word_tokenize(text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]


    # Stemming instead of lemmatization
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(word) for word in tokens]

    # Join tokens back into a single string
    return ' '.join(stemmed_tokens)

# Assuming 'data' is your DataFrame and 'review_body' is the column you want to clean
data['cleaned_review_body'] = data['review_body'].apply(preprocess_text)

In [ ]:
data['cleaned_review_body'].head(10)

#Text Represnation


In [ ]:
X = data['cleaned_review_body']
y = data['sentiment']

In [ ]:
X

In [ ]:
y

In [ ]:
vectorizer = TfidfVectorizer()
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = vectorizer.fit_transform(X_train_text)
x_test = vectorizer.transform(X_test_text)


#Split data


In [ ]:
# The split and training-only vocabulary fit are performed above.


In [ ]:
print("Vectorized x_train shape:", X_train.shape)
print("Vectorized x_test shape:", x_test.shape)


#Naive Bayes Model

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)


In [ ]:
y_pred_nb = nb_model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred_nb)
print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

conf_matrix = confusion_matrix(y_test, y_pred_nb)

fig, ax = plt.subplots()
cax = ax.matshow(conf_matrix, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Adding color bar
fig.colorbar(cax)

# Add annotations of the confusion matrix numbers
for (i, j), val in np.ndenumerate(conf_matrix):
    ax.text(j, i, f'{val}', ha='center', va='center', color='black')

# Set tick marks for grid
ax.set_xticks([0, 1], labels=['Negative', 'Positive'])
ax.set_yticks([0, 1], labels=['Negative', 'Positive'])

plt.show()

In [ ]:
# Function to preprocess, vectorize text, and predict sentiment
def predict_sentiment(texts):
    # Define the mapping between numerical labels and sentiment strings
    label_to_sentiment = {0: 'Negative', 1: 'Positive'}  # Adjust if your labels are different
    # Preprocess the user input
    texts = [preprocess_text(text) for text in texts]
    # Transform the user input using the fitted TF-IDF vectorizer
    transformed_texts = vectorizer.transform(texts)
    # Making predictions using the Naive Bayes model
    predictions = nb_model.predict(transformed_texts)
    # Convert numerical predictions back to string labels manually
    predicted_labels = [label_to_sentiment[pred] for pred in predictions]
    for text, label in zip(texts, predicted_labels):
        print(f'Text: {text} --> Sentiment: {label}')

# Taking input from the user
user_input = input("Please enter your text: ")
predict_sentiment([user_input])

#Random Forset


In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)


In [ ]:
y_pred_rf = rf_model.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred_rf))


#Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

conf_matrix = confusion_matrix(y_test, y_pred_rf)

fig, ax = plt.subplots()
cax = ax.matshow(conf_matrix, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Adding color bar
fig.colorbar(cax)

# Add annotations of the confusion matrix numbers
for (i, j), val in np.ndenumerate(conf_matrix):
    ax.text(j, i, f'{val}', ha='center', va='center', color='black')

# Set tick marks for grid
ax.set_xticks([0, 1], labels=['Negative', 'Positive'])
ax.set_yticks([0, 1], labels=['Negative', 'Positive'])

plt.show()

In [ ]:
# Function to preprocess, vectorize text, and predict sentiment
def predict_sentiment(texts):
    # Define the mapping between numerical labels and sentiment strings
    label_to_sentiment = {0: 'Negative', 1: 'Positive'}  # Adjust if your labels are different
    # Preprocess the user input
    texts = [preprocess_text(text) for text in texts]
    # Transform the user input using the fitted TF-IDF vectorizer
    transformed_texts = vectorizer.transform(texts)
    # Making predictions using the Naive Bayes model
    predictions = rf_model.predict(transformed_texts)
    # Convert numerical predictions back to string labels manually
    predicted_labels = [label_to_sentiment[pred] for pred in predictions]
    for text, label in zip(texts, predicted_labels):
        print(f'Text: {text} --> Sentiment: {label}')

# Taking input from the user
user_input = input("Please enter your text: ")
predict_sentiment([user_input])